# Cryptocurrency Wallet Risk Classification - Model Training Pipeline

This notebook trains a Graph Neural Network (GNN) model to classify cryptocurrency wallets as benign or criminal using the REAL-CATS dataset.

## Pipeline Overview

1. **Data Preparation**: Load and split REAL-CATS dataset (benign/criminal wallets)
2. **Feature Engineering**: Create behavioral features (flow_ratio, fan_ratio)
3. **Graph Construction**: Fetch transaction data from mempool.space API
4. **Tensor Building**: Create PyTorch Geometric graph tensors
5. **Model Training**: Train GAT-based GNN model
6. **Evaluation**: Test model performance

## Requirements

- REAL-CATS dataset files in `../Real_Cats_data/`:
  - `BB.tsv` - Benign Bitcoin wallets
  - `CB.tsv` - Criminal Bitcoin wallets
- Internet connection for mempool.space API access
- Installed packages: torch, torch-geometric, pandas, scikit-learn, requests

## Note

**Transaction fetching takes significant time** due to API rate limits (1 req/sec). For testing, reduce the sample size in the merge step.

In [14]:
#IMPORTS
import pandas as pd
import numpy as np
import os
import torch
import requests
import time
from sklearn.preprocessing import StandardScaler
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv
from sklearn.model_selection import train_test_split

# --- CONFIGURATION ---
# Set PROJECT_ROOT to the parent of the current folder (models/)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
FULL_DATASET_DIR = os.path.join(PROJECT_ROOT, 'Real_Cats_data', 'full_dataset')
REAL_CATS_DATA_DIR = os.path.join(PROJECT_ROOT, 'Real_Cats_data')
PATH_BENIGN   = os.path.join(REAL_CATS_DATA_DIR, 'BB.tsv')                # Your Clean Data
PATH_CRIMINAL = os.path.join(REAL_CATS_DATA_DIR, 'wallets_behavioral.tsv') # Your Criminal Data
PATH_CB       = os.path.join(REAL_CATS_DATA_DIR, 'CB.tsv')
os.makedirs(FULL_DATASET_DIR, exist_ok=True)

# THE WHITELIST: The specific columns we want the model to learn from.
# Any column NOT in this list (like addresses, IDs) will be dropped automatically later.
NUMERIC_FEATURES = [
    'balance', 
    'total_received_USD', 
    'total_sent_USD', 
    'transaction_fee', 
    'transaction_fee_Variance',
    'max_sent_amount', 
    'min_sent_amount',
    'lifetime', 
    'total_output_slots', 
    'total_input_slots', 
    'activity_w', 
    'activity_d', 
    'activity_time',
    'transaction_number',
    'payment_transactions',
    'receipt_transactions',
    'received_Variance_USD',
    'sent_Variance_USD'
]

print("Configuration loaded. Feature list defined.")
print(f"PATH_CB: {PATH_CB}")
print(f"REAL_CATS_DATA_DIR: {REAL_CATS_DATA_DIR}")


Configuration loaded. Feature list defined.
PATH_CB: d:\Projects\final_project\Real_Cats_data\CB.tsv
REAL_CATS_DATA_DIR: d:\Projects\final_project\Real_Cats_data


In [5]:
def split_criminal_wallet(input_path=f"{REAL_CATS_DATA_DIR}/CB.tsv", output_dir=f"{REAL_CATS_DATA_DIR}/"):

    df = pd.read_csv(input_path, sep='\t', low_memory=False)
    behavioral_wallets = df[
        (df["transaction_number"] > 0) |
        (df["total_received_BTC"] > 0) |
        (df["total_sent_BTC"] > 0)
    ].copy()

    non_behavioral_wallets = df[
        (df["transaction_number"] == 0) &
        (df["total_received_BTC"] == 0) &
        (df["total_sent_BTC"] == 0)
    ].copy()

    print("Total wallets:", len(df))
    print("Behavioral:", len(behavioral_wallets))
    print("Non-behavioral:", len(non_behavioral_wallets))

    behavioral_wallets.to_csv(f"{output_dir}/wallets_behavioral.tsv", sep='\t', index=False)
    non_behavioral_wallets.to_csv(f"{output_dir}/wallets_non_behavioral.tsv", sep='\t', index=False)

    print("Saved wallets_behavioral and non-behavioral wallets to", output_dir)
    return behavioral_wallets, non_behavioral_wallets

split_criminal_wallet()

Total wallets: 90597
Behavioral: 40032
Non-behavioral: 50565
Saved wallets_behavioral and non-behavioral wallets to d:\Projects\final_project\Real_Cats_data/


(                                          address              label  \
 0               111KvKxkeia8NeKMzqEDqnGm1v49Ncp3j     Blackmail Scam   
 1              11212qhtzpz2SugohG3xk1FCWY9U5TG8mg         Ransomware   
 4              1122NYbAT2KkZDZ5TFvGy4D2Ut7eYfx4en         Ransomware   
 6              1123Hubtw2CXGqFKvvozUcbWe6SbTT5ydg         Ransomware   
 14             1128Ev6iMRQ25SuGAnrekSqbW8aQpX5PZQ    Investment Scam   
 ...                                           ...                ...   
 90583  bc1qzzaxpxts27jxcq842fazy7sqvuese0njt5qrfv              Other   
 90585  bc1qzzdgdj3tzv9fsww3p9e2jcm5pzemh9uawv0cus  Social Media Scam   
 90586  bc1qzznxeqlcvzey9nkaeaffv64pna8n8tpnr8l6nn               Hack   
 90590  bc1qzzx3esv0a0he9nkmjxza3alfycgx4jhgqullz9      Giveaway Scam   
 90591  bc1qzzx8fmg85w33608lqwh7unh6yk9nxyjsgxy2pq              Other   
 
         balance  total_received_BTC  total_sent_BTC  total_received_USD  \
 0           0.0            180000.0        18

In [6]:
def load_and_label_datasets():
    """Load benign and criminal datasets and add labels"""
    print("1. Loading and Labeling Data...")
    
    # A. Load Benign (Label = 0)
    try:
        df_b = pd.read_csv(PATH_BENIGN, sep="\t")
        df_b['label'] = 0
        print(f"   Loaded {len(df_b)} Benign records.")
    except Exception as e:
        print(f"   Error loading Benign: {e}")
        df_b = pd.DataFrame() # Return empty if fails

    # B. Load Criminal (Label = 1)
    try:
        df_c = pd.read_csv(PATH_CRIMINAL, sep="\t")
        df_c['label'] = 1
        print(f"   Loaded {len(df_c)} Criminal records.")
    except Exception as e:
        print(f"   Error loading Criminal: {e}")
        df_c = pd.DataFrame()

    return df_b, df_c

a, b = load_and_label_datasets()
print("Is 'address' a column in a?", 'address' in a.columns)
print("Is 'address' a column in b?", 'address' in b.columns)

1. Loading and Labeling Data...
   Loaded 90164 Benign records.
   Loaded 40032 Criminal records.
Is 'address' a column in a? True
Is 'address' a column in b? True


In [7]:
def merge_datasets(df_benign, df_criminal):
    """Merge benign and criminal datasets, keeping only common columns"""
    print("2. Merging Datasets...")
    
    # We only keep columns that both files share
    common_cols = list(set(df_benign.columns) & set(df_criminal.columns))
    df_merged = pd.concat([df_benign[common_cols], df_criminal[common_cols]], ignore_index=True)
    df_merged = df_merged.sample(frac=1, random_state=42).reset_index(drop=True)
    df_merged = df_merged.head(100).copy()
    print(f"   Common columns: {len(common_cols)}")
    print(f"   Total Records Merged: {len(df_merged)}")
    return df_merged

c = merge_datasets(a, b)
print("Is 'address' a column in c?", 'address' in c.columns)

2. Merging Datasets...
   Common columns: 36
   Total Records Merged: 100
Is 'address' a column in c? True


In [8]:
def perform_feature_engineering(df):
    print("2b. Engineering Features (Cleaning & Creating Ratios)...")
    
    # 1. Create Ratios (Safe division to avoid /0 error)
    # Flow Ratio: Are they a pass-through entity?
    # We use (received + 1e-5) to prevent crashing on 0 division
    df['flow_ratio'] = df['total_sent_USD'] / (df['total_received_USD'] + 1e-5)
    
    # Fan Ratio: One-to-Many (Distribution) vs Many-to-One (Aggregation)
    df['fan_ratio'] = df['total_output_slots'] / (df['total_input_slots'] + 1e-5)
    
    # 2. Select the "Whitelist" of useful features
    # These are the ones we KEEP for the GNN.
    # We prioritize USD over BTC, and Variances over raw mins/maxes.
    final_features = [
        # VOLUME (Size of operation)
        'balance', 
        'total_received_USD', 
        'total_sent_USD',
        
        # VELOCITY (Speed/Intensity)
        'lifetime',
        'transaction_number',
        'activity_w',
        'activity_d',
        'activity_time', # Distinct active hours/periods
        
        # BEHAVIOR (Structure)
        'transaction_fee',
        'transaction_fee_Variance', # Crucial: Automated vs Manual
        'received_Variance_USD',    # Crucial: Fixed vs Random ransom amounts
        'sent_Variance_USD',
        'total_input_slots',
        'total_output_slots',
        'payment_transactions',     # Outgoing count
        'receipt_transactions',     # Incoming count
        
        # ENGINEERED (The Ratios)
        'flow_ratio',
        'fan_ratio'
    ]
    
    print(f"   Selected {len(final_features)} features for the Model.")
    return df, final_features

In [9]:
def fetch_edges_mempool_directed(df):
    print("2. Fetching DIRECTED Graph Connections (Source: Mempool.space)...")
    edges_list = []
    headers = {'User-Agent': 'Mozilla/5.0'}
    
    for i, row in df.iterrows():
        my_address = row['address']
        print(f"   [{i+1}/10] Processing: {my_address}...")
        
        # Parse Time Window
        try:
            start_ts = pd.to_datetime(row['first_time']).timestamp()
            end_ts = pd.to_datetime(row['last_time']).timestamp()
        except:
            start_ts, end_ts = 0, 9999999999

        try:
            url = f"https://mempool.space/api/address/{my_address}/txs"
            r = requests.get(url, headers=headers, timeout=10)
            
            if r.status_code == 200:
                txs = r.json()
                
                for tx in txs:
                    # 1. Check Time
                    if not tx.get('status', {}).get('confirmed'): continue
                    tx_time = tx['status']['block_time']
                    if tx_time < start_ts or tx_time > end_ts: continue
                    
                    # 2. Identify My Role (Sender or Receiver?)
                    
                    # --- CHECK OUTGOING (Am I an Input?) ---
                    is_sender = False
                    for inp in tx.get('vin', []):
                        if inp.get('prevout', {}).get('scriptpubkey_address') == my_address:
                            is_sender = True
                            break
                    
                    if is_sender:
                        # I am the SOURCE. I connect to all Outputs.
                        for out in tx.get('vout', []):
                            recipient = out.get('scriptpubkey_address')
                            amount = out.get('value', 0) # Amount in Satoshis
                            
                            if recipient and recipient != my_address:
                                edges_list.append({
                                    'source': my_address,
                                    'target': recipient,
                                    'weight': amount, 
                                    'timestamp': tx_time,
                                    'direction': 'outgoing'
                                })

                    # --- CHECK INCOMING (Am I an Output?) ---
                    # Note: You can be both sender and receiver (Change address), 
                    # but we excluded self-loops above (recipient != my_address).
                    
                    # Find out how much I received specifically
                    amount_received = 0
                    is_receiver = False
                    for out in tx.get('vout', []):
                        if out.get('scriptpubkey_address') == my_address:
                            amount_received += out.get('value', 0)
                            is_receiver = True
                    
                    if is_receiver:
                        # I am the TARGET. All Inputs connect to me.
                        for inp in tx.get('vin', []):
                            sender = inp.get('prevout', {}).get('scriptpubkey_address')
                            
                            if sender and sender != my_address:
                                edges_list.append({
                                    'source': sender,
                                    'target': my_address,
                                    'weight': amount_received, # We attribute the full receive amount to the link
                                    'timestamp': tx_time,
                                    'direction': 'incoming'
                                })
                                
            time.sleep(1) # Be polite
            
        except Exception as e:
            print(f"       !! Error: {e}")

    df_edges = pd.DataFrame(edges_list)
    print(f"\n   DONE. Found {len(df_edges)} Directed Edges.")
    print(df_edges.head())
    return df_edges


In [10]:
def process_and_save_tensors(df_nodes, df_edges):
    print("\n4. Building Tensors (with Log Scaling)...")
    
    # A. Clean Edges
    if not df_edges.empty:
        df_edges = df_edges.drop_duplicates(subset=['source', 'target', 'timestamp'])
        # Log Scale Edge Weights (Handling potential negatives just in case)
        w = pd.to_numeric(df_edges['weight'], errors='coerce').fillna(0)
        df_edges['weight_log'] = np.log1p(np.maximum(0, w))
    else:
        df_edges = pd.DataFrame(columns=['source', 'target', 'weight_log'])

    # B. Map Addresses
    known_addrs = df_nodes['address'].tolist()
    ghost_addrs = list(set(df_edges['source']).union(set(df_edges['target'])) - set(known_addrs))
    all_nodes = known_addrs + ghost_addrs
    addr_map = {addr: i for i, addr in enumerate(all_nodes)}
    
    print(f"   Nodes: {len(all_nodes)} ({len(known_addrs)} Known + {len(ghost_addrs)} Ghosts)")

    # C. Build X (Features)
    print(f"   Scaling {len(NUMERIC_FEATURES)} features...")
    df_scaled = df_nodes.copy()
    
    for c in NUMERIC_FEATURES:
        if c in df_scaled.columns:
            # 1. Force Numeric & Fill NA
            series = pd.to_numeric(df_scaled[c], errors='coerce').fillna(0)
            
            # 2. Clip Negatives (Safety) & Log Scale
            # We use maximum(0, x) because log(-1) is NaN
            df_scaled[c] = np.log1p(np.maximum(0, series))
        else:
            df_scaled[c] = 0.0

    # D. Standardization (Z-Score)
    scaler = StandardScaler()
    known_feats = scaler.fit_transform(df_scaled[NUMERIC_FEATURES])
    
    # E. Create Matrix
    x_np = np.zeros((len(all_nodes), len(NUMERIC_FEATURES)))
    x_np[0:len(known_addrs)] = known_feats
    x = torch.tensor(x_np, dtype=torch.float)
    
    # F. Labels
    y_np = np.full(len(all_nodes), -1)
    y_np[0:len(known_addrs)] = df_nodes['label'].values
    y = torch.tensor(y_np, dtype=torch.long)
    
    # G. Edges
    if not df_edges.empty:
        src = df_edges['source'].map(addr_map).values
        dst = df_edges['target'].map(addr_map).values
        
        mask = (~np.isnan(src)) & (~np.isnan(dst))
        edge_index = torch.tensor([src[mask], dst[mask]], dtype=torch.long)
        edge_attr = torch.tensor(df_edges['weight_log'].to_numpy()[mask].reshape(-1, 1), dtype=torch.float)
    else:
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_attr = torch.empty((0, 1), dtype=torch.float)

    # Save
    torch.save(x, os.path.join(FULL_DATASET_DIR, 'x.pt'))
    torch.save(y, os.path.join(FULL_DATASET_DIR, 'y.pt'))
    torch.save(edge_index, os.path.join(FULL_DATASET_DIR, 'edge_index.pt'))
    torch.save(edge_attr, os.path.join(FULL_DATASET_DIR, 'edge_attr.pt'))
    
    print(f"   Saved Tensors. X shape: {x.shape}")

In [11]:
class CryptoGNN(torch.nn.Module):
    def __init__(self, num_node_features, num_edge_features, hidden_channels, num_classes):
        super().__init__()
        self.conv1 = GATv2Conv(num_node_features, hidden_channels, heads=2, edge_dim=num_edge_features)
        self.conv2 = GATv2Conv(hidden_channels * 2, hidden_channels, heads=1, edge_dim=num_edge_features)
        self.classifier = torch.nn.Linear(hidden_channels, num_classes)

    def forward(self, x, edge_index, edge_attr):
        h = self.conv1(x, edge_index, edge_attr=edge_attr)
        h = h.relu()
        h = F.dropout(h, p=0.3, training=self.training)
        h = self.conv2(h, edge_index, edge_attr=edge_attr)
        h = h.relu()
        return self.classifier(h)

In [12]:
def train_and_test():
    print("\n5. Training Model...")
    # Load
    x = torch.load(os.path.join(FULL_DATASET_DIR, 'x.pt'))
    y = torch.load(os.path.join(FULL_DATASET_DIR, 'y.pt'))
    edge_index = torch.load(os.path.join(FULL_DATASET_DIR, 'edge_index.pt'))
    edge_attr = torch.load(os.path.join(FULL_DATASET_DIR, 'edge_attr.pt'))

    # Split (Only known nodes)
    valid_idx = torch.where(y != -1)[0].numpy()
    if len(valid_idx) > 1:
        train_idx, test_idx = train_test_split(valid_idx, test_size=0.2, stratify=y[valid_idx])
    else:
        train_idx, test_idx = valid_idx, valid_idx

    # Initialize
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"   Using device: {device}")
    
    model = CryptoGNN(x.shape[1], edge_attr.shape[1], 16, 2).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = torch.nn.CrossEntropyLoss()
    
    # Move data to device
    x = x.to(device)
    y = y.to(device)
    edge_index = edge_index.to(device)
    edge_attr = edge_attr.to(device)

    # Train
    model.train()
    for epoch in range(51):
        optimizer.zero_grad()
        out = model(x, edge_index, edge_attr)
        loss = criterion(out[train_idx], y[train_idx])
        loss.backward()
        optimizer.step()
        if epoch % 10 == 0: print(f"   Epoch {epoch} | Loss: {loss.item():.4f}")

    # Test
    model.eval()
    with torch.no_grad():
        out = model(x, edge_index, edge_attr)
        pred = out.argmax(dim=1)
        correct = (pred[test_idx] == y[test_idx]).sum()
        acc = int(correct) / len(test_idx)
        print(f"   Test Accuracy: {acc:.2f}")
    
    # Save model
    model_path = os.path.join(PROJECT_ROOT, 'models', 'crypto_gnn_model.pt')
    torch.save(model, model_path)
    print(f"\n   Model saved to: {model_path}")
    
    return model

In [ ]:
# --- EXECUTION PIPELINE ---
print("="*60)
print("Starting Training Pipeline")
print("="*60)

# 1. Split Data (Creates wallets_behavioral.tsv and wallets_non_behavioral.tsv)
split_criminal_wallet()

# 2. Load & Label (Robustly loads the correct files)
df_benign, df_criminal = load_and_label_datasets()

# 3. Merge (Robustly merges them)
df = merge_datasets(df_benign, df_criminal)

# Optional: Use small sample for testing (UNCOMMENT to use full dataset)
df = df.head(100)
print(f"\n[INFO] Using sample of {len(df)} wallets for faster testing")
print("[INFO] To use full dataset, comment out the line above")

# Save the sample for future reference
sample_path = os.path.join(REAL_CATS_DATA_DIR, 'sample_100_wallets.tsv')
df.to_csv(sample_path, sep='\t', index=False)
print(f"[INFO] Sample saved to: {sample_path}")

if not df.empty:
    # 4. Feature Engineering (The NEW step)
    # We clean the data and define the exact list of columns to use
    df, NUMERIC_FEATURES = perform_feature_engineering(df)

    # 5. Fetch Graph
    # This uses your high-quality mempool logic
    print(f"\n[WARNING] About to fetch transaction data for {len(df)} wallets")
    print("[WARNING] This will take approximately {:.1f} minutes (1 req/sec rate limit)".format(len(df)/60))
    
    # Prompt user before fetching (only works in interactive mode)
    # response = input("Continue? (y/n): ")
    # if response.lower() != 'y':
    #     print("Pipeline aborted by user")
    # else:
    df_edges = fetch_edges_mempool_directed(df)

    # 6. Build Tensors (Now using the correct NUMERIC_FEATURES list)
    process_and_save_tensors(df, df_edges)

    # 7. Train
    model = train_and_test()
    
    print("\n" + "="*60)
    print("Training Complete!")
    print("="*60)
else:
    print("Pipeline aborted: No data found.")

Starting Training Pipeline
Total wallets: 90597
Behavioral: 40032
Non-behavioral: 50565
Saved wallets_behavioral and non-behavioral wallets to d:\Projects\final_project\Real_Cats_data/
1. Loading and Labeling Data...
   Loaded 90164 Benign records.
   Loaded 40032 Criminal records.
2. Merging Datasets...
   Common columns: 36
   Total Records Merged: 100

[INFO] Using sample of 100 wallets for faster testing
[INFO] To use full dataset, comment out the line above
2b. Engineering Features (Cleaning & Creating Ratios)...
   Selected 18 features for the Model.

[WARNING] About to fetch transaction data for 100 wallets
[WARNING] This will take approximately 1.7 minutes (1 req/sec rate limit)
2. Fetching DIRECTED Graph Connections (Source: Mempool.space)...
   [1/10] Processing: 1F4yn3gLAGDXfg5mS1w4oUmN9HhRXHrHqN...
   [2/10] Processing: 14kCwWrLQNv4JZPUeYeJ1R8RxysY8MAUBn...
   [3/10] Processing: 18xM1Zi8FdsjX1bAgujKAjVNsFaLZNiEw...
   [4/10] Processing: 1EUdsRr1Ca9ntwwbCUXJicsTgfyAmSdch7...

C:\Users\orimood\AppData\Local\Temp\ipykernel_18708\3476960862.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_edges['weight_log'] = np.log1p(np.maximum(0, w))
C:\Users\orimood\AppData\Local\Temp\ipykernel_18708\3476960862.py:56: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  edge_index = torch.tensor([src[mask], dst[mask]], dtype=torch.long)


   Saved Tensors. X shape: torch.Size([5195, 18])

5. Training Model...
   Using device: cpu
   Epoch 0 | Loss: 0.7403
   Epoch 10 | Loss: 0.6158
   Epoch 20 | Loss: 0.5686
   Epoch 30 | Loss: 0.5313
   Epoch 40 | Loss: 0.5215
   Epoch 50 | Loss: 0.4845
   Test Accuracy: 0.70

   Model saved to: d:\Projects\final_project\models\crypto_gnn_model.pt

Training Complete!


In [15]:
# Save the current df sample to file
sample_path = os.path.join(REAL_CATS_DATA_DIR, 'sample_100_wallets.tsv')
df.to_csv(sample_path, sep='\t', index=False)
print(f"Sample of {len(df)} wallets saved to: {sample_path}")

Sample of 100 wallets saved to: d:\Projects\final_project\Real_Cats_data\sample_100_wallets.tsv
